# ArmorVault — One-click OCR-VL test

This is the simple version: no Google Drive, no uploads, and no real documents. It runs PaddleOCR-VL through the official Transformers/PyTorch runtime in a short-lived worker process, then runs MiniCPM-V on the T4 GPU after that process exits. It does not install PaddlePaddle.

Select **Runtime → Change runtime type → T4 GPU**, then choose **Run all**.

In [ ]:
import os, subprocess, sys
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
except Exception as exc:
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU first.') from exc
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
%pip install -q 'transformers==4.57.6' 'accelerate>=1.4' 'bitsandbytes==0.49.2' pillow
print('Dependencies installed.')

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from PIL import Image
import json, subprocess, sys, time

root = Path('/content/armorvault-one-click')
root.mkdir(exist_ok=True)
image_path = root / 'public_demo.png'
urlretrieve('https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/paddleocr_vl_demo.png', image_path)
display(Image.open(image_path))

# PaddleOCR-VL via Transformers runs in a separate process and exits before MiniCPM is loaded.
worker = r'''
import sys, inspect, torch
from pathlib import Path
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor
from transformers import masking_utils
input_path, output_path = Path(sys.argv[1]), Path(sys.argv[2])
output_path.mkdir(parents=True, exist_ok=True)
# PaddleOCR-VL's remote code uses the older inputs_embeds name; newer Transformers uses input_embeds.
_create_causal_mask = masking_utils.create_causal_mask
_mask_params = inspect.signature(_create_causal_mask).parameters
if 'input_embeds' in _mask_params and 'inputs_embeds' not in _mask_params:
    def _compat_create_causal_mask(*args, inputs_embeds=None, input_embeds=None, **kwargs):
        if input_embeds is None:
            input_embeds = inputs_embeds
        return _create_causal_mask(*args, input_embeds=input_embeds, **kwargs)
    masking_utils.create_causal_mask = _compat_create_causal_mask
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Colab T4 is compute capability 7.5; float16 is supported reliably, while bfloat16 can fail inside vision attention.
dtype = torch.float16 if device == 'cuda' else torch.float32
model_path = 'PaddlePaddle/PaddleOCR-VL'
prompts = {'ocr': 'OCR:', 'table': 'Table Recognition:', 'formula': 'Formula Recognition:', 'chart': 'Chart Recognition:'}
model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True, torch_dtype=dtype).to(device).eval()
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
messages = [{'role': 'user', 'content': [{'type': 'image', 'image': Image.open(input_path).convert('RGB')}, {'type': 'text', 'text': prompts['ocr']}]}]
inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt').to(device)
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False, use_cache=True)
text = processor.batch_decode(outputs, skip_special_tokens=True)[0]
(output_path / 'extraction.txt').write_text(text, encoding='utf-8')
print('PaddleOCR-VL Transformers complete')
'''
worker_path = root / 'paddle_worker.py'
paddle_output = root / 'paddle-output'
paddle_output.mkdir(exist_ok=True)
worker_path.write_text(worker, encoding='utf-8')
started = time.perf_counter()
completed = subprocess.run([sys.executable, str(worker_path), str(image_path), str(paddle_output)], text=True, capture_output=True)
diagnostics = (completed.stdout + '\n' + completed.stderr).strip()
if diagnostics:
    print('PaddleOCR-VL diagnostics:\n' + diagnostics)
if completed.returncode != 0:
    raise RuntimeError('PaddleOCR-VL stopped before producing output (exit ' + str(completed.returncode) + '). Full diagnostics:\n' + diagnostics[-12000:])
extraction_path = paddle_output / 'extraction.txt'
if not extraction_path.exists():
    raise RuntimeError('PaddleOCR-VL finished without creating extraction.txt.')
print({'paddleSeconds': round(time.perf_counter() - started, 2), 'extraction': str(extraction_path)})

In [ ]:
import json, subprocess, sys, time
from pathlib import Path
extraction_path = root / 'paddle-output' / 'extraction.txt'
if not extraction_path.exists():
    raise RuntimeError('The OCR stage did not create extraction.txt. Run the notebook from the top; do not run this cell alone.')
minicpm_output = root / 'minicpm-output'
minicpm_output.mkdir(parents=True, exist_ok=True)
worker = r'''
import json, re, sys, time, torch
from pathlib import Path
from PIL import Image
from transformers import AutoModel, AutoTokenizer
input_path, extraction_path, output_path = map(Path, sys.argv[1:4])
model_name = 'openbmb/MiniCPM-V-4_5-int4'
extraction = extraction_path.read_text(encoding='utf-8', errors='replace')[:60000]
started = time.perf_counter()
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, device_map='auto', low_cpu_mem_usage=True).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
init_seconds = time.perf_counter() - started
prompt = """Read this public test document using the image and PaddleOCR-VL extraction. Return JSON only with exactly these keys: documentType, documentNumber, holderName, issueDate, expiryDate, totalAmount, currency, mrzLines. Use null for uncertain scalar values and [] for absent MRZ lines. Dates must be YYYY-MM-DD. Never invent unreadable characters.\n\nPaddleOCR-VL extraction:\n""" + extraction
started = time.perf_counter()
answer = model.chat(msgs=[{'role': 'user', 'content': [Image.open(input_path).convert('RGB'), prompt]}], tokenizer=tokenizer, enable_thinking=False)
inference_seconds = time.perf_counter() - started
cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', answer.strip(), flags=re.I | re.S)
try:
    fields = json.loads(cleaned)
except json.JSONDecodeError:
    start, end = cleaned.find('{'), cleaned.rfind('}')
    if start < 0 or end <= start:
        raise RuntimeError('MiniCPM did not return a JSON object.')
    fields = json.loads(cleaned[start:end + 1])
result = {'extractor': 'PaddleOCR-VL-0.9B', 'understandingModel': model_name, 'miniCPMInitializationSeconds': round(init_seconds, 2), 'miniCPMInferenceSeconds': round(inference_seconds, 2), 'fields': fields}
(output_path / 'result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(result, ensure_ascii=False, indent=2))
'''
worker_path = root / 'minicpm_worker.py'
worker_path.write_text(worker, encoding='utf-8')
# Use a fresh child process so cached Colab imports cannot retain an older bitsandbytes build.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-deps', 'transformers==4.51.3', 'tokenizers==0.21.4', 'bitsandbytes==0.49.2'])
started = time.perf_counter()
completed = subprocess.run([sys.executable, str(worker_path), str(image_path), str(extraction_path), str(minicpm_output)], text=True, capture_output=True)
diagnostics = (completed.stdout + '\n' + completed.stderr).strip()
if diagnostics:
    print('MiniCPM-V diagnostics:\n' + diagnostics)
if completed.returncode != 0:
    raise RuntimeError('MiniCPM-V stopped before producing result.json (exit ' + str(completed.returncode) + '). Full diagnostics:\n' + diagnostics[-16000:])
result_path = minicpm_output / 'result.json'
if not result_path.exists():
    raise RuntimeError('MiniCPM-V finished without creating result.json.')
print({'miniCPMSeconds': round(time.perf_counter() - started, 2)})
print(result_path.read_text(encoding='utf-8'))

The final result is saved inside the temporary Colab session at `/content/armorvault-one-click/result.json`. No Google Drive permission is requested.